In [0]:
bronze_path = "/Volumes/workspace/ecommerce/ecommerce_data/medallion/bronze/events"
silver_path = "/Volumes/workspace/ecommerce/ecommerce_data/medallion/silver/events"
gold_path   = "/Volumes/workspace/ecommerce/ecommerce_data/medallion/gold"


In [0]:
events_raw = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/",
    header=True,
    inferSchema=True
)

events_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)


In [0]:
spark.read.format("delta").load(bronze_path).printSchema()


In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)


In [0]:
from pyspark.sql import functions as F

silver_df = (
    bronze_df
    .filter(F.col("event_type").isNotNull())
    .filter(F.col("price").isNotNull())
    .filter(F.col("user_id").isNotNull())
    .dropDuplicates(["user_session", "event_time"])
)


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)


In [0]:
spark.read.format("delta").load(silver_path).count()


In [0]:
silver_df = spark.read.format("delta").load(silver_path)


In [0]:
gold_brand_revenue = (
    silver_df
    .filter(F.col("event_type") == "purchase")
    .groupBy("brand")
    .agg(F.sum("price").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

gold_brand_revenue.show(5)


In [0]:
gold_conversion = (
    silver_df
    .groupBy("category_code", "event_type")
    .count()
    .groupBy("category_code")
    .pivot("event_type")
    .sum("count")
    .withColumn(
        "conversion_rate",
        (F.col("purchase") / F.col("view")) * 100
    )
)


In [0]:
gold_brand_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/brand_revenue")

gold_conversion.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/conversion_rate")


In [0]:
new_data = events_raw.limit(1000)


In [0]:
new_data.write \
    .format("delta") \
    .mode("append") \
    .save(bronze_path)


In [0]:
bronze_updated = spark.read.format("delta").load(bronze_path)

silver_updated = (
    bronze_updated
    .filter(F.col("event_type").isNotNull())
    .filter(F.col("price").isNotNull())
    .dropDuplicates(["user_session", "event_time"])
)

silver_updated.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)
